# Video Frame Extractor

**Input:** `Google Drive/MyDrive/mmaudio-outputs/` (videolar)
**Output:** `Google Drive/MyDrive/mmaudio-outputs/` (aynı klasör, her videonun yanına JPG/PNG)

`mmaudio-outputs` klasöründeki videolardan ilk kareyi çıkarır ve her videonun yanına (aynı klasöre, aynı isimle) kaydeder.

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
import os
from pathlib import Path

import cv2
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
DRIVE_OUTPUT_FOLDER = "mmaudio-outputs"
drive_output_path = f"/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}"

FORMAT = "jpg"        # "jpg" or "png"
OVERWRITE = False     # True = re-extract even if photo already exists

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def is_video(path: Path) -> bool:
    return path.suffix.lower() in VIDEO_EXTENSIONS


def find_videos(root: Path) -> list[Path]:
    if root.is_file():
        return [root] if is_video(root) else []
    return sorted(p for p in root.rglob("*") if p.is_file() and is_video(p))


def extract_first_frame_bytes(video_path: Path, fmt: str = "jpg") -> bytes | None:
    cap = cv2.VideoCapture(str(video_path))
    try:
        if not cap.isOpened():
            return None
        ret, frame = cap.read()
        if not ret or frame is None:
            return None
        ok, buf = cv2.imencode(f".{fmt}", frame)
        return buf.tobytes() if ok else None
    finally:
        cap.release()

In [ ]:
root = Path(drive_output_path)
videos = find_videos(root)

print(f"Klasör: {root}")
print(f"Bulunan video: {len(videos)}")
for v in videos[:5]:
    print(f"  {v.relative_to(root)}")
if len(videos) > 5:
    print(f"  ... (+{len(videos) - 5} more)")

In [ ]:
succeeded = 0
skipped = 0
failed = 0
last_out_path: Path | None = None

for video in videos:
    rel = video.relative_to(root)
    out_path = video.with_suffix(f".{FORMAT}")

    if out_path.exists() and not OVERWRITE:
        print(f"  SKIP  {rel}")
        skipped += 1
        continue

    data = extract_first_frame_bytes(video, FORMAT)
    if data is None:
        print(f"  FAIL  {rel}")
        failed += 1
        continue

    os.makedirs(out_path.parent, exist_ok=True)
    out_path.write_bytes(data)
    last_out_path = out_path
    print(f"  OK    {rel}")
    succeeded += 1

print()
print(f"Done — {succeeded} extracted, {skipped} skipped, {failed} failed")
print(f"Output: {root}")

In [ ]:
from IPython.display import Image, display

if last_out_path is not None:
    print(last_out_path.name)
    display(Image(filename=str(last_out_path)))
else:
    print("Bu çalıştırmada yeni çıkarılan frame yok.")